# RAG Pipeline Projects Tour

End-to-end walkthrough of both RAG pipeline projects built on `rag_common`.

| Project | Corpus | Ground truth | Embedding |
|---|---|---|---|
| **P3** `rag_pipeline_systematic_evals` | Single PDF (119 pages) | Synthetic QA per chunk config | OpenAI API |
| **P4** `rag_pipeline_experimentation` | Multi-doc (50–1000 PDFs) | Real qrels (Open RAG Benchmark) | SentenceTransformers (local) |

**No API keys required.** Live demos use a deterministic stub embedder; charts load real results from disk.

```
rag_common              ← models, chunkers, retrievers, metrics, parsers, base ABCs
├── P3 (systematic)     ← 24-cell grid, synthetic QA, OpenAI embeddings
└── P4 (experimentation)← RAGPipeline, multi-doc, real qrels, SentenceTransformers
```

In [ ]:
import json
import sys
import textwrap
from pathlib import Path
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project roots from this notebook's location:
# rag_common/notebooks/rag_pipelines_tour.ipynb
PROJECTS_ROOT = Path().resolve().parent.parent   # .../newline/projects
P3_DIR = PROJECTS_ROOT / "rag_pipeline_systematic_evals"
P4_DIR = PROJECTS_ROOT / "rag_pipeline_experimentation"

# Only add P4 to sys.path — P3 results are loaded from JSON to avoid
# a naming collision between the two projects' src/config.py modules.
if str(P4_DIR) not in sys.path:
    sys.path.insert(0, str(P4_DIR))

P3_EXP_DIR = P3_DIR / "experiments"
P4_EXP_DIR = P4_DIR / "experiments" / "results"

print(f"Projects root : {PROJECTS_ROOT}")
print(f"P3 results    : {len(list(P3_EXP_DIR.glob('*.json')))} experiments on disk")
print(f"P4 results    : {len(list(P4_EXP_DIR.glob('*.json')))} experiments on disk")

In [ ]:
# Deterministic stub embedder — no API key, no model download.
# Hash-seeded unit vectors, identical across runs.

DIM = 64

def stub_embed(texts: list[str]) -> np.ndarray:
    vecs = []
    for t in texts:
        seed = abs(hash(t[:40])) % (2**31)
        v = np.random.default_rng(seed).standard_normal(DIM).astype(np.float32)
        vecs.append(v / np.linalg.norm(v))
    return np.array(vecs)

print("stub_embed(['a', 'b']).shape =", stub_embed(["a", "b"]).shape)

---

## 1. Shared Foundation — `rag_common`

Both projects import from `rag_common`. This section shows the additions made for P4: two new chunkers and the ABC layer. See [`rag_common_tour.ipynb`](rag_common_tour.ipynb) for the full foundation walkthrough.

### 1a. All five chunkers — now in `rag_common`

In [ ]:
from rag_common.chunkers import (
    FixedSizeChunker, SentenceBasedChunker, SemanticChunker,
    RecursiveChunker, SlidingWindowChunker,
)

TEXT = textwrap.dedent("""\
    Retrieval-Augmented Generation combines a retriever and a generator.
    The retriever finds relevant passages from a large document corpus.
    The generator produces a grounded answer using those passages as context.
    Chunking strategy directly affects which passages are retrieved.
    Fixed-size chunks are fast but can split mid-sentence or mid-idea.
    Recursive chunking tries paragraph boundaries first, then smaller separators.
    Semantic chunking groups sentences until the topic changes detectably.
    Sliding-window chunks maximise overlap to improve recall at the cost of index size.
    Embedding models map text to dense vectors for similarity search.
    BM25 provides complementary sparse keyword matching.
    Hybrid retrieval fuses both scores after min-max normalisation.
""")

strategies = {
    "FixedSize(200,ol=40)":    FixedSizeChunker(chunk_size=200, overlap=40),
    "Sentence(3s,ol=1)":       SentenceBasedChunker(sentences_per_chunk=3, overlap_sentences=1),
    "Semantic(thresh=0.5)":    SemanticChunker(embed_fn=stub_embed, breakpoint_threshold=0.5),
    "Recursive(200,ol=40)":    RecursiveChunker(chunk_size=200, overlap=40),
    "SlidingWindow(w=4,s=2)":  SlidingWindowChunker(window_size=4, step=2),
}

rows = []
for name, chunker in strategies.items():
    chunks = chunker.chunk(TEXT)
    rows.append({
        "strategy":  name,
        "used_by":   "P3+P4" if "Sliding" not in name and "Recursive" not in name else "P4",
        "n_chunks":  len(chunks),
        "avg_chars": int(np.mean([len(c.content) for c in chunks])),
        "method":    chunks[0].method,
        "example":   chunks[0].content[:55].replace("\n", " ") + "…",
    })

pd.DataFrame(rows).set_index("strategy")

### 1b. RecursiveChunker — why separator hierarchy matters

Splits on `["\n\n", "\n", ". ", " "]` in order. Unlike `FixedSizeChunker`, it never cuts inside a paragraph when a paragraph boundary is available — critical for preserving research paper section structure.

In [ ]:
multi_para = textwrap.dedent("""\
    First paragraph introduces the research problem.
    It spans multiple sentences with supporting evidence.

    Second paragraph describes the proposed method.
    Several design choices are explained and justified here.

    Third paragraph presents experimental results.
""")

print("=== FixedSizeChunker(120, ol=20) ===")
for c in FixedSizeChunker(120, 20).chunk(multi_para):
    print(f"  [{c.chunk_index}] {c.content!r}")

print("\n=== RecursiveChunker(120, ol=20) ===")
for c in RecursiveChunker(120, 20).chunk(multi_para):
    print(f"  [{c.chunk_index}] {c.content!r}")

### 1c. Abstract base classes — `rag_common.base`

P4 introduces ABCs so any conforming implementation is swappable via config without touching evaluation code.

In [ ]:
from rag_common.base import BaseChunker, BaseEmbedder, BaseRetriever, BaseReranker, BaseLLM
from src.embedders import SentenceTransformersEmbedder

print("ABCs defined in rag_common.base:")
for cls in [BaseChunker, BaseEmbedder, BaseRetriever, BaseReranker, BaseLLM]:
    print(f"  {cls.__name__}")

print()
print(f"SentenceTransformersEmbedder is BaseEmbedder: {issubclass(SentenceTransformersEmbedder, BaseEmbedder)}")
print(f"RecursiveChunker has .chunk():                {hasattr(RecursiveChunker, 'chunk')}")

---

## 2. P3 — Systematic Evaluation (`rag_pipeline_systematic_evals`)

P3 runs a fixed **4×2×3 grid** (24 experiments) on a single PDF (`fy10syb.pdf` — US DoJ FY2010 Immigration Statistical Yearbook, 119 pages). For each of the 4 chunking configs, a **separate synthetic QA dataset** is generated and used as ground truth.

```
4 ChunkConfigs  ×  2 EmbedModels             ×  3 RetrievalMethods  =  24 experiments
fixed_256        text-embedding-3-small (1536d)   vector (FAISS cosine)
fixed_512        text-embedding-3-large (3072d)   bm25  (BM25Okapi)
sentence_5s                                       hybrid (α=0.5)
semantic_t0.65
```

### 2a. Grid structure

In [ ]:
# Load all 24 results from disk as raw JSON — no P3 src import needed.
p3_raw = [
    json.loads(p.read_text())
    for p in sorted(P3_EXP_DIR.glob("*.json"))
]
print(f"Loaded {len(p3_raw)} P3 experiment results\n")

# Show the first 6 to illustrate the cross-product structure
pd.DataFrame([
    {
        "experiment_id": r["experiment_id"],
        "chunk":         r["config"]["chunk"]["strategy"],
        "embed":         r["config"]["embed"]["model"].split("-")[-1],  # "small" / "large"
        "retrieval":     r["config"]["retrieval"]["method"],
        "n_qa_pairs":    r["config"]["qa_pairs_per_config"],
    }
    for r in p3_raw[:6]
])

### 2b. Why per-config QA datasets?

This is the most important design decision in P3. Each chunking config produces different chunk UUIDs. Using a shared QA dataset across configs would produce invalid metrics — the chunk IDs in the ground truth wouldn't match the retrieved chunk IDs.

```
config_A → chunks_A (UUIDs) → qa_dataset_A (ground truth = chunk UUID from A)
config_B → chunks_B (UUIDs) → qa_dataset_B (ground truth = chunk UUID from B)

✓  evaluate config_A against qa_dataset_A   ← valid
✗  evaluate config_B against qa_dataset_A   ← chunk IDs won't match → MRR=0
```

**Invalidation rule**: cache is valid only when stored chunk IDs are a subset of the current chunk set. Any re-parse or re-chunk that changes UUIDs triggers regeneration.

### 2c. Leaderboard — all 24 results

In [ ]:
def _p3_row(r: dict) -> dict:
    m = r["metrics"]
    return {
        "experiment_id": r["experiment_id"],
        "chunk":   r["config"]["chunk"]["strategy"],
        "embed":   r["config"]["embed"]["model"].split("-")[-1],
        "retrieval": r["config"]["retrieval"]["method"],
        "mrr":       round(m["mrr"], 4),
        "recall@5": round(m["recall_at_k"]["5"], 4),
        "ndcg@5":   round(m["ndcg_at_k"]["5"], 4),
        "latency_ms": round(m["avg_retrieval_time_s"] * 1000, 1),
    }

p3_df = (
    pd.DataFrame([_p3_row(r) for r in p3_raw])
    .sort_values("mrr", ascending=False)
    .reset_index(drop=True)
)
p3_df.head(8)

### 2d. Configuration dimension impact

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in [
    (axes[0], "chunk",     "Chunk Strategy"),
    (axes[1], "embed",     "Embedding Model"),
    (axes[2], "retrieval", "Retrieval Method"),
]:
    grouped = p3_df.groupby(col)["mrr"].mean().sort_values(ascending=False)
    bars = grouped.plot(kind="bar", ax=ax, color="#4C72B0", width=0.6)
    ax.set_title(f"Avg MRR by {title}", fontsize=11)
    ax.set_ylabel("MRR")
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.suptitle("P3: Impact of each configuration dimension on MRR (fy10syb.pdf)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2e. MRR heatmap — chunk strategy × embedding model

In [ ]:
pivot = p3_df.pivot_table(values="mrr", index="chunk", columns="embed", aggfunc="mean")

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("P3: MRR — Chunk Strategy × Embedding Model", pad=12)
ax.set_xlabel("Embedding Model")
ax.set_ylabel("Chunk Strategy")
plt.tight_layout()
plt.show()

best = p3_df.iloc[0]
print(f"Best overall:  {best['experiment_id']}")
print(f"  MRR={best['mrr']:.4f}  Recall@5={best['recall@5']:.4f}  NDCG@5={best['ndcg@5']:.4f}")
print()
print("Key finding: semantic chunking dominates on dense statistical government text.")
print("fixed_256 underperforms — splits mid-sentence inside tightly-packed tables.")
print("Vector retrieval beats hybrid — BM25 is weak on numeric/tabular content.")

### 2f. Sample query from the best experiment

In [ ]:
best_exp_path = P3_EXP_DIR / f"{best['experiment_id']}.json"
best_data = json.loads(best_exp_path.read_text())
sample = best_data["query_results"][0]

print(f"Experiment : {best['experiment_id']}")
print(f"Question   : {sample['question']}")
print(f"Hit rank   : {'1' if sample['retrieved_ids'][0] in sample['relevant_ids'] else '>1'}")
print(f"MRR        : {sample['rr']:.4f}")
print(f"Recall@5   : {sample['recall@5']:.4f}")

---

## 3. P4 — Experimentation Pipeline (`rag_pipeline_experimentation`)

P4 extends the pattern to multiple documents and real ground truth. The key architectural difference: **`RAGPipeline`** encapsulates ingest + query as one object, backed by a persisted FAISS index spanning all documents.

```
INGESTION:  PDFs → parse (PyMuPDF) → chunk → embed → FAISS index → save to disk
QUERY:      question → embed → retrieve → [rerank] → list[RetrievalResult]
EVALUATION: qrels.json → doc-level ground truth → rag_common.metrics
```

Grid: `3 chunking × 2 embedding × 2 retrieval = 12 baseline cells`

### 3a. RAGPipeline — ingest and query (stub embedder, no PDF required)

In [ ]:
import tempfile
from src.pipeline import RAGPipeline
from src.base import BaseEmbedder
from rag_common.chunkers import RecursiveChunker

class StubEmbedder(BaseEmbedder):
    """Stub that satisfies BaseEmbedder using the hash-based stub_embed."""
    @property
    def model_name(self): return "stub-64d"
    @property
    def dimension(self): return DIM
    def embed(self, texts): return stub_embed(texts)
    def embed_chunks(self, chunks, chunk_label=""):
        return self.embed([c.content for c in chunks])

# Build one pipeline per retrieval method to compare
pipelines = {
    method: RAGPipeline(
        chunker=RecursiveChunker(chunk_size=300, overlap=60),
        embedder=StubEmbedder(),
        retrieval_method=method,
        alpha=0.6,
    )
    for method in ["dense", "bm25", "hybrid"]
}
print("RAGPipeline instances created:", list(pipelines.keys()))

In [ ]:
# Simulate two-paper corpus — patch _parse_pdf so no real PDFs needed
CORPUS = {
    "attention.pdf": (
        "Attention mechanisms allow models to weigh relevance of each input token. "
        "Transformers use multi-head self-attention over keys, queries, and values. "
        "Scaled dot-product attention divides by the square root of the key dimension "
        "to prevent vanishing gradients in deep networks."
    ),
    "retrieval.pdf": (
        "Dense retrieval embeds queries and documents into a shared vector space. "
        "BM25 uses term frequency and inverse document frequency for sparse ranking. "
        "Hybrid systems combine dense and sparse scores with a tunable alpha weight. "
        "Reranking applies a cross-encoder as a second-stage precision filter."
    ),
}

def fake_parse(path): return CORPUS[path.name], 2

query = "how does attention mechanism work in transformers"
results_by_method = {}

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    pdf_paths = []
    for name in CORPUS:
        p = tmp_path / name
        p.write_bytes(b"%PDF-1.4")
        pdf_paths.append(p)

    with patch("src.pipeline._parse_pdf", side_effect=fake_parse):
        for method, pipeline in pipelines.items():
            pipeline.ingest(pdf_paths, index_dir=tmp_path / f"idx_{method}")
            results_by_method[method] = pipeline.query(query, top_k=3)

print(f"Query: {query!r}\n")
for method, results in results_by_method.items():
    print(f"[{method}]")
    for r in results:
        print(f"  score={r.score:.4f}  doc={r.chunk.document_id}  {r.chunk.content[:55]!r}")
    print()

### 3b. Ground truth — real qrels vs synthetic QA

| | P3 | P4 |
|---|---|---|
| Source | LLM-generated (gpt-4o-mini + Instructor) | Vectara Open RAG Benchmark |
| Granularity | **Chunk-level** (UUID) | **Document-level** (paper ID) |
| Size | 25 QA pairs per chunking config | 3,045 QA pairs across 1,000 papers |
| Format | `{question, chunk_id}` | `{query_id: {doc_id, section_id}}` |

The evaluator maps `retrieved chunk IDs → chunk.document_id`, then compares against the qrels document ID. This is coarser than chunk-level evaluation but scales to large multi-doc corpora without manual labeling.

In [ ]:
qrels_path = P4_DIR / "data" / "qrels_filtered.json"
if qrels_path.exists():
    qrels = json.loads(qrels_path.read_text())
    paper_ids = {v["relevant_doc_ids"][0] for v in qrels.values()}
    print(f"qrels_filtered.json: {len(qrels)} queries across {len(paper_ids)} papers\n")
    for qid, entry in list(qrels.items())[:3]:
        print(f"  {qid[:8]}…")
        print(f"    query           : {entry['query'][:70]!r}")
        print(f"    relevant_doc_ids: {entry['relevant_doc_ids']}")
        print()
else:
    print("qrels_filtered.json not found — run: python scripts/download_dataset.py --limit 50")

### 3c. P4 experiment results — 12-cell POC grid

In [ ]:
p4_raw = [
    json.loads(p.read_text())
    for p in sorted(P4_EXP_DIR.glob("*.json"))
]

def _p4_row(r: dict) -> dict:
    cfg = r["config"]
    return {
        "experiment_id": r["experiment_id"],
        "chunk":     cfg["chunk"]["strategy"],
        "embed":     cfg["embed"]["model"].replace("all-", "").split("-v")[0],
        "retrieval": cfg["retrieval"]["method"],
        "mrr":       round(r["metrics"].get("mrr", 0), 4),
        "recall@5":  round(r["metrics"].get("recall@5", 0), 4),
        "ndcg@5":    round(r["metrics"].get("ndcg@5", 0), 4),
        "n_queries": r["n_queries"],
        "latency_ms": round(r["avg_latency_s"] * 1000, 1),
    }

p4_df = (
    pd.DataFrame([_p4_row(r) for r in p4_raw])
    .sort_values("mrr", ascending=False)
    .reset_index(drop=True)
)
print(f"Loaded {len(p4_df)} P4 results")
p4_df

### 3d. P4: Configuration dimension impact

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in [
    (axes[0], "chunk",     "Chunk Strategy"),
    (axes[1], "embed",     "Embedding Model"),
    (axes[2], "retrieval", "Retrieval Method"),
]:
    grouped = p4_df.groupby(col)["mrr"].mean().sort_values(ascending=False)
    grouped.plot(kind="bar", ax=ax, color="#55A868", width=0.6)
    ax.set_title(f"Avg MRR by {title}", fontsize=11)
    ax.set_ylabel("MRR")
    ax.set_ylim(0, 1.15)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.suptitle("P4: Configuration impact on MRR (5-paper POC corpus, 9 queries)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("MRR=1.0 across the board is expected on a 5-paper corpus.")
print("Discrimination emerges at 50+ papers where doc-level retrieval becomes harder.")

---

## 4. P3 vs P4 — Side-by-Side Comparison

In [ ]:
pd.DataFrame([
    {"Aspect": "Corpus",             "P3": "Single PDF (119 pages)",              "P4": "Multi-doc (50–1,000 PDFs)"},
    {"Aspect": "PDF parser",         "P3": "pdfplumber",                           "P4": "PyMuPDF (rag_common.parsers)"},
    {"Aspect": "Ground truth",       "P3": "Synthetic QA per chunk config",        "P4": "Real qrels (Open RAG Benchmark)"},
    {"Aspect": "GT granularity",     "P3": "Chunk-level (UUID)",                   "P4": "Document-level (paper ID)"},
    {"Aspect": "Embedding",          "P3": "OpenAI API (text-embedding-3-*)",      "P4": "SentenceTransformers (local, free)"},
    {"Aspect": "Chunkers",           "P3": "FixedSize, Sentence, Semantic",        "P4": "+ Recursive, SlidingWindow"},
    {"Aspect": "Grid size",          "P3": "4×2×3 = 24 experiments",               "P4": "3×2×2 = 12 baseline cells"},
    {"Aspect": "Component ABCs",     "P3": "No (duck-typed callables)",            "P4": "Yes (rag_common.base)"},
    {"Aspect": "Multi-doc indexing", "P3": "No (one PDF)",                         "P4": "Yes (RAGPipeline.ingest)"},
    {"Aspect": "Reranking",          "P3": "cross-encoder (optional)",             "P4": "cross-encoder (optional)"},
    {"Aspect": "Generation layer",   "P3": "No",                                   "P4": "Planned (LiteLLM + citations)"},
    {"Aspect": "Web UI",             "P3": "No",                                   "P4": "Planned (Streamlit)"},
    {"Aspect": "Tests",              "P3": "105 unit tests (all mocked)",           "P4": "56 unit tests + smoke test"},
]).set_index("Aspect")

### 4a. Shared `rag_common.metrics` — same call, different ID semantics

In [ ]:
from rag_common import metrics

# P3: chunk UUIDs as IDs (1 relevant per query, always)
p3_qr = [
    (["chunk-A", "chunk-B", "chunk-C"], {"chunk-A"}),
    (["chunk-X", "chunk-A", "chunk-C"], {"chunk-X"}),
    (["chunk-B", "chunk-X", "chunk-A"], {"chunk-B"}),
]

# P4: paper IDs as IDs (can have multiple relevant docs)
p4_qr = [
    (["paper-001", "paper-002", "paper-003"], {"paper-001"}),
    (["paper-004", "paper-001", "paper-002"], {"paper-004"}),
    (["paper-002", "paper-004", "paper-001"], {"paper-002"}),
]

# Identical metrics.evaluate() call regardless of what the IDs represent
p3_scores = metrics.evaluate(p3_qr, k=5)
p4_scores = metrics.evaluate(p4_qr, k=5)

pd.DataFrame({
    "P3 (chunk-level GT)": p3_scores,
    "P4 (doc-level GT)":   p4_scores,
}).round(3)

### 4b. What lives where

In [ ]:
layout = {
    "rag_common/models.py":      ("Chunk, RetrievalResult",          "P3 + P4"),
    "rag_common/chunkers.py":    ("FixedSize, Sentence, Semantic, Recursive, SlidingWindow", "P3 + P4"),
    "rag_common/retrievers.py":  ("BM25, Dense, Hybrid",             "P3 + P4"),
    "rag_common/metrics.py":     ("recall@k, precision@k, mrr, ndcg","P3 + P4"),
    "rag_common/vector_store.py":("FAISSVectorStore",                 "P3 + P4"),
    "rag_common/parsers.py":     ("parse_pdf() via PyMuPDF",         "P4 (usable by P3)"),
    "rag_common/base.py":        ("BaseChunker/Embedder/Retriever/…","P4 ABCs"),
    "P3/src/config.py":          ("ChunkConfig, EmbedModel, ExperimentConfig, build_experiment_grid()", "P3"),
    "P3/src/embedders.py":       ("OpenAI batch embedder + tiktoken + cache", "P3"),
    "P3/src/qa_generator.py":    ("Instructor + gpt-4o-mini → synthetic QA", "P3"),
    "P3/src/grid_search.py":     ("24-cell orchestrator (resume + force)",   "P3"),
    "P3/src/visualizer.py":      ("6 Matplotlib/Seaborn charts",              "P3"),
    "P4/src/config.py":          ("ChunkConfig, EmbedModelName (ST names), build_experiment_grid()", "P4"),
    "P4/src/embedders.py":       ("SentenceTransformersEmbedder + cache",     "P4"),
    "P4/src/pipeline.py":        ("RAGPipeline (ingest + query + save/load)", "P4"),
    "P4/src/evaluator.py":       ("doc-level qrels runner",                   "P4"),
    "P4/src/experiment.py":      ("12-cell orchestrator (resume + force)",    "P4"),
}

pd.DataFrame(
    [{"Module": k, "Exports": v[0], "Used by": v[1]} for k, v in layout.items()]
).set_index("Module")

---

## 5. Next Steps

| Project | Status | What's next |
|---|---|---|
| P3 | Complete — 24/24 cells, 105 tests, blog written | Optional: Cohere reranker, extra charts |
| P4 | POC done — 12-cell smoke grid, 56 tests | Full 50-paper baseline; generator + judge; Streamlit UI |

**Run P4 full baseline grid (102 queries, 41 papers):**
```bash
cd rag_pipeline_experimentation
python scripts/evaluate.py data/papers/ data/qrels_filtered.json \\
    --config config/experiments/baseline.yaml --top-k 5
```

**Run smoke test (2 papers, 3 queries, 1 cell, ~16s):**
```bash
python scripts/evaluate.py data/smoke_papers/ data/qrels_smoke.json \\
    --config config/experiments/smoke.yaml --top-k 1 --force
```

**Run P3 (single PDF, 24 experiments, ~30 min with OpenAI):**
```bash
cd rag_pipeline_systematic_evals
python -m src.main data/fy10syb.pdf
```